In [1]:

import pandas as pd

df = pd.read_csv('bitcoin_dataset.csv', index_col=0)

df['benefit'] = df['Close'] - df['Open']
df['class'] = (df['benefit'] > 0).astype(int)

df['y'] = df['class'].shift(-1)
df['y_reg'] = df['Close'].shift(-1)

df = df.dropna()
features = ['Open', 'High', 'Low', 'Close', 'Volume']
X = df[features]
y = df['y'].astype(int)
y_reg = df['y_reg']

print(f"Dataset ready. Total samples: {len(df)}")

Dataset ready. Total samples: 1460


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)
_, _, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42, shuffle=False)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_reg_scaled = scaler_y.fit_transform(y_train_reg.values.reshape(-1, 1)).ravel()



In [3]:

import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.metrics import accuracy_score, classification_report

nn_clf = Sequential([
    Input(shape=(5,)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])
nn_clf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

start_time = time.time()
nn_clf.fit(X_train_scaled, y_train, epochs=30, batch_size=16, verbose=0)
time_nn_clf = time.time() - start_time

y_pred_nn_clf = (nn_clf.predict(X_test_scaled, verbose=0) > 0.5).astype(int).ravel()
acc_nn_clf = accuracy_score(y_test, y_pred_nn_clf)

print(f"Neural Net Classification Time: {time_nn_clf:.4f} seconds")
print(f"Accuracy: {acc_nn_clf * 100:.2f}%\n")
print(classification_report(y_test, y_pred_nn_clf))

Neural Net Classification Time: 4.1533 seconds
Accuracy: 50.68%

              precision    recall  f1-score   support

           0       0.50      0.95      0.66       146
           1       0.56      0.07      0.12       146

    accuracy                           0.51       292
   macro avg       0.53      0.51      0.39       292
weighted avg       0.53      0.51      0.39       292



In [4]:

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.metrics import mean_squared_error

nn_reg = Sequential([
    Input(shape=(5,)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1)
])
nn_reg.compile(optimizer='adam', loss='mse')

start_time = time.time()
nn_reg.fit(X_train_scaled, y_train_reg_scaled, epochs=30, batch_size=16, verbose=0)
time_nn_reg = time.time() - start_time

preds_scaled = nn_reg.predict(X_test_scaled, verbose=0)
preds_reg = scaler_y.inverse_transform(preds_scaled).ravel()
rmse_nn_reg = np.sqrt(mean_squared_error(y_test_reg, preds_reg))

print(f"Neural Net Regression Time: {time_nn_reg:.4f} seconds")
print(f"RMSE: ${rmse_nn_reg:.2f}")

Neural Net Regression Time: 3.8781 seconds
RMSE: $785.52


In [5]:

from xgboost import XGBClassifier, XGBRegressor

t0 = time.time()
xgb_c = XGBClassifier(random_state=42, eval_metric='logloss').fit(X_train, y_train)
time_classic_clf = time.time() - t0
acc_classic_clf = accuracy_score(y_test, xgb_c.predict(X_test))

t0 = time.time()
xgb_r = XGBRegressor(random_state=42).fit(X_train, y_train_reg)
time_classic_reg = time.time() - t0
rmse_classic_reg = np.sqrt(mean_squared_error(y_test_reg, xgb_r.predict(X_test)))

In [6]:

comparison_df = pd.DataFrame({
    'Task': ['Classification', 'Classification', 'Regression', 'Regression'],
    'Model': ['XGBoost (Week 8)', 'MLP Neural Net (Week 9)', 'XGBoost (Week 8)', 'MLP Neural Net (Week 9)'],
    'Time (s)': [round(time_classic_clf, 4), round(time_nn_clf, 4), round(time_classic_reg, 4), round(time_nn_reg, 4)],
    'Metric': [f"Accuracy: {acc_classic_clf*100:.2f}%", f"Accuracy: {acc_nn_clf*100:.2f}%", f"RMSE: ${rmse_classic_reg:.2f}", f"RMSE: ${rmse_nn_reg:.2f}"]
})

print("====================== Benchmark Report ======================")
print(comparison_df.to_string(index=False))

====================== Benchmark Report ======================
          Task                   Model  Time (s)           Metric
Classification        XGBoost (Week 8)    0.4349 Accuracy: 52.74%
Classification MLP Neural Net (Week 9)    4.1533 Accuracy: 50.68%
    Regression        XGBoost (Week 8)    0.1691   RMSE: $1675.55
    Regression MLP Neural Net (Week 9)    3.8781    RMSE: $785.52
